# Recurrent Neural Networks (RNNs): The Complete Guide to Sequential Logic

Welcome to the definitive guide on RNNs. In this notebook, we bridge the gap between static neural networks (like CNNs/MLPs) and dynamic sequence models.

---

## 1. Why do we need RNNs? (The Failure of Dense Networks)

If we already have powerful Multi-Layer Perceptrons (MLPs), why do we need an entirely new architecture for text or stock prices?

### A. The Fixed-Size Problem
Standard Dense networks (MLPs) require a **fixed input size**.
*   **The Scenario:** You want to classify the sentiment of a sentence.
*   **The Conflict:** Sentence A is "I love this" (3 words). Sentence B is "This is the most incredible experience of my life" (9 words).
*   **The Dense "Hack":** You could force a max length of 10 and pad shorter sentences with zeros. But what if a sentence has 100 words? You lose information.

### B. The Memory Problem
Dense networks have no concept of **Time** or **Order**. They treat every input as a single, isolated "snapshot." In a sequence, the same word "Bank" means something different in "river bank" vs "central bank." Context is temporal.

![Dense vs RNN Architecture](assets/dnn_vs_rnn.gif)

---

## 2. The Core Issues of RNNs: Why they are hard to train

While RNNs are theoretically brilliant, they suffer from three major "real-world" problems that an engineer must understand deeply.

### Issue 1: Short-Term Memory (Long-Term Dependency)
**What it is:** The network becomes very good at remembering what it just saw 2-3 steps ago, but it "forgets" the beginning of a sequence.
**Why it happens:** In an RNN, the hidden state $h_t$ is updated at every single step: $h_t = f(h_{t-1}, x_t)$. This is like a game of **Telephone**.
*   At step 1, you have a clear message.
*   By step 50, that message has been whispered through 50 different "updates."
*   Unless the weights are perfect, the original meaning of the first word gets "washed out" or overwritten by more recent words.
**Consequence:** For long sentences, the network might forget the subject of the sentence by the time it reaches the verb.

### Issue 2: The Computational Bottleneck
**What it is:** RNNs are slow to train, even on powerful GPUs.
**The GPU Paradox:** GPUs are fast because they can do thousands of things at once (**Parallelization**). CNNs love GPUs because you can calculate all pixels in an image simultaneously.
**The RNN Problem:** You **cannot** calculate the hidden state $h_t$ until you have finished calculating $h_{t-1}$. 
*   This is a **Sequential Dependency**.
*   Your $2,000 GPU is forced to sit idle, waiting for one single calculation to finish before it can start the next one.
*   **Result:** You cannot parallelize across the time dimension.

### Issue 3: Vanishing/Exploding Gradients
**What it is:** During Backpropagation, the signal used to update the weights either becomes zero (Vanishing) or incredibly huge (Exploding).
**The Math Intuition:** training an RNN involves multiplying the same weight matrix $W$ over and over for every time step.
*   If $W = 0.9$ and you have 100 steps: $(0.9)^{100} \approx 0.00002$ (The signal dies).
*   If $W = 1.1$ and you have 100 steps: $(1.1)^{100} \approx 13,780$ (The signal explodes).

---

## 3. Interview Corner: How to explain Vanishing Gradients without Math

If an interviewer asks you to explain the Vanishing Gradient problem without writing equations, use the **"Photocopy of a Photocopy"** analogy:

> "Imagine you have a clear piece of paper (the error signal). You make a photocopy of it. Then you take that copy and make a photocopy of it. If the copier settings aren't perfect ($W < 1$), each version gets a little more blurry. By the 50th copy, you have a blank piece of white paper.
>
> In an RNN, because we use the **exact same weights** for every step, we are essentially 'photocopying' the signal through time. If the weights are small, the signal fades to zero (Vanishes), and the early layers 'never see' the error, so they never learn to improve."

---

## 4. Components of an RNN: The Story of Three Weights

An RNN isn't just one layer; it's a recurrence. To understand the computation, look at the three weight matrices that tell the story:

1.  **$W_{xh}$ (Input-to-Hidden):** Learns how much the **current input** ($x_t$) should influence the memory.
2.  **$W_{hh}$ (Hidden-to-Hidden):** Learns how much the **previous memory** ($h_{t-1}$) should be preserved or updated.
3.  **$W_{hy}$ (Hidden-to-Output):** Learns how to translate the internal memory into a final prediction ($y_t$).

![Unfolding the RNN](assets/rnn_unfold.gif)

---

## 5. Computation Flow: Uni-directional vs. Bi-directional

### Uni-directional RNN (The Classic)
The standard RNN only looks at the **past**. It reads left-to-right. 
*   **Best for:** Time-series forecasting, real-time speech-to-text.
*   **Limitation:** It doesn't know what's coming next.

### Bi-directional RNN (The Context King)
What if the 3rd word in a sentence depends on the 5th word? (e.g., "The **crane**... **lifted** the beam" vs "The **crane**... **flew** away").
*   **How it works:** It runs two RNNs—one Forward and one Backward.
*   **The Result:** The output at time $t$ is a concatenation of both directions: $H_t = [\overrightarrow{h}_t ; \overleftarrow{h}_t]$.

![Bidirectional RNN Flow](assets/rnn_bidirectional.gif)

**Use Case:** Sentiment Analysis, Named Entity Recognition (NER), Machine Translation.


## 6. The Mathematical Proof: The Vanishing Gradient Problem

Why does an RNN "forget" the beginning of a long sentence? Let's prove it with a simple derivation of the gradient.

### The Recurrence
Recall the hidden state update: $h_t = \sigma(W_{hh} h_{t-1} + W_{xh} x_t)$
For simplicity, let's assume a linear activation ($\sigma(x) = x$).

### The Gradient Chain
To update the weights $W_{hh}$, we need to calculate the gradient of the loss $L$ at time $T$ with respect to the weights used at time $k$ (where $k \ll T$).
By the chain rule:
$$ \frac{\partial L_T}{\partial W_{hh}} = \frac{\partial L_T}{\partial h_T} \times \frac{\partial h_T}{\partial h_k} \times \frac{\partial h_k}{\partial W_{hh}} $$

Focus on the bridge term: $\frac{\partial h_T}{\partial h_k}$
$$ \frac{\partial h_T}{\partial h_k} = \prod_{i=k+1}^T \frac{\partial h_i}{\partial h_{i-1}} $$

Since $h_i = W_{hh} h_{i-1} + \dots$, the derivative $\frac{\partial h_i}{\partial h_{i-1}} = W_{hh}$.
Therefore:
$$ \frac{\partial h_T}{\partial h_k} = (W_{hh})^{T-k} $$

### The "Death" of the Gradient
1.  **If $|W_{hh}| < 1$:** As the sequence length ($T-k$) gets large, $(W_{hh})^{T-k}$ approaches **zero** exponentially. The gradient **vanishes**. The network learns nothing from the past.
2.  **If $|W_{hh}| > 1$:** The gradient approaches **infinity** exponentially. The training **explodes** and crashes.

![Vanishing Gradient Animation](assets/rnn_vanishing.gif)


In [ ]:
import torch
import torch.nn as nn

# --- END-TO-END WALKTHROUGH: Sentiment Analysis (Many-to-One) ---

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(SentimentRNN, self).__init__()
        # 1. Embedding Layer: Turns word IDs into dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # 2. RNN Layer: Processes the sequence
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        
        # 3. Linear Head: Many-to-One (we use both directions of the final hidden state)
        self.fc = nn.Linear(hidden_dim * 2, 1) # *2 for Bidirectional
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (Batch, SeqLen)
        embedded = self.embedding(x) # (Batch, SeqLen, EmbedDim)
        
        # rnn_out: (Batch, SeqLen, Hidden*2)
        # hidden: (2, Batch, Hidden) -> [forward_final, backward_final]
        rnn_out, hidden = self.rnn(embedded)
        
        # Concatenate final forward (hidden[0]) and final backward (hidden[1])
        final_context = torch.cat((hidden[0], hidden[1]), dim=1)
        
        return self.sigmoid(self.fc(final_context))

# Simulation
model = SentimentRNN(vocab_size=1000, embed_dim=16, hidden_dim=32)
example_input = torch.randint(0, 1000, (1, 5)) # A 5-word sentence
prediction = model(example_input)

print(f"Input Shape (5 words): {example_input.shape}")
print(f"Sentiment Score (0-1): {prediction.item():.4f}")


## 7. Summary of Issues & Edge Cases

| Issue | Reason | Solution |
| :--- | :--- | :--- |
| **Vanishing Gradient** | Repeated multiplication of weights $< 1$ in the BPTT chain. | LSTMs, GRUs, or Gradient Clipping. |
| **Exploding Gradient** | Repeated multiplication of weights $> 1$. | Gradient Clipping (hard cap on max value). |
| **Short-Term Memory** | The "Hidden State" gets overwritten by new information too easily. | Gated units (Input/Forget gates). |
| **Computational Bottleneck** | Steps must be computed sequentially ($t$ depends on $t-1$). You cannot fully parallelize RNNs on a GPU like you can with CNNs. | Transformers (Self-Attention). |

### Edge Case: The "Empty Input" or "Zero Padding"
In practice, we often pad sequences with zeros. A naive RNN will treat those zeros as "real data" and update the hidden state, potentially "forgetting" the actual sentence that came before the zeros. 
**Solution:** Use `torch.nn.utils.rnn.pack_padded_sequence` to tell PyTorch to skip the zeros during computation.
